# fit_cpc_model.ipynb





In [ ]:
# import hssm
# import hssm.plotting
# import numpy as np
# import pandas as pd
# import arviz as az
# import seaborn as sns
# from matplotlib import pyplot as plt
# import bambi as bmb
# import statsmodels.api as sm
# #from pymer4.models import Lmer
# from scipy import stats
# from scipy.stats import pearsonr
# from matplotlib.patches import Patch
# from matplotlib.colors import to_rgb
# from scipy.interpolate import UnivariateSpline
# import matplotlib.patheffects as pe
# from statsmodels.stats.outliers_influence import variance_inflation_factor
# from numpy.polynomial.legendre import legvander
# import statsmodels.formula.api as smf
# from scipy.stats import norm

import hssm
import numpy as np
import pandas as pd
import arviz as az
from matplotlib import pyplot as plt
from pathlib import Path
#%pip install ydata-profiling
#from ydata_profiling import ProfileReport




Path("model_outputs_07.09_cpc").mkdir(exist_ok=True)

In [ ]:

#pwd

## Load prepped data

In [ ]:
pair_pre = pd.read_csv("hssm_ema_df_pre_07.09.csv")

# make sure subject is string 
pair_pre["subject"] = pair_pre["subject"].astype(str)

# make sure subject is in seconds 
pair_pre["rt"] = pair_pre["rt"].astype(float) / 1000


# make sure response is -1 and 1 / rt is numeric / check range of delta_sb_within_subj_normed / check values of delta_valence / check type and range of affect_z
vars_to_check = ["subject", "response", "rt", "delta_sb_within_subj_normed", "delta_valence", "affect_z"]

summary = pd.DataFrame({
    "dtype": [pair_pre[v].dtype for v in vars_to_check],
    "n_nan": [pair_pre[v].isna().sum() for v in vars_to_check],
    "min": [pair_pre[v].min() if pair_pre[v].dtype != "object" else "—" for v in vars_to_check],
    "max": [pair_pre[v].max() if pair_pre[v].dtype != "object" else "—" for v in vars_to_check],
    "unique": [sorted(pair_pre[v].unique())[:5] if pair_pre[v].nunique() < 10 else f"{pair_pre[v].nunique()} values" for v in vars_to_check],
}, index=vars_to_check)

print(summary)

print(f"PRE:  {len(pair_pre)} trials, {pair_pre['subject'].nunique()} subjects")

print("PRE:", pair_pre.columns.tolist())


## FIT MODEL FOR PRE-TRIP DATA 


In [ ]:
# MAKE PRE TRIP DATA MODEL 
# in hssm models use delta_sb within subject normalized
cpc_model = hssm.HSSM(
    model="ddm",
    data=pair_pre,
    include=[
        {
            "name": "v",
            "formula": "v ~ 1 + delta_sb_within_subj_normed + delta_valence*affect_z + (1 + delta_sb_within_subj_normed + delta_valence*affect_z | subject)",
            #"formula": "v ~ 1 + r_diff_norm + delta_valence*affect_z + (1 + delta_sb_normed + delta_valence + affect_z | subject)",

        }
    ],
)

cpc_model

## RUN  MODEL 

In [ ]:
ddm_samples_cpc_model = cpc_model.sample(
    sampler = "mcmc",
    tune = 2000, # warmpup samples
    draws = 4000, # 4000
    chains = 4, 
    cores = 4,
    target_accept=0.95,
    #idata_kwargs=dict(log_likelihood=True)  # return log likelihood
    mp_ctx="spawn"
)

ddm_samples_cpc_model.to_netcdf('model_outputs_07.09_cpc/cpc_model.nc')

print("Over")

## Print model

In [ ]:
ddm_samples_cpc_model

## Graph